# Review Live/Paper Trading Run

Visualize a paper or live trading run using the same charts as the backtest
notebooks — TradingView Lightweight Charts with trade markers, equity curve,
stats summary.

Data comes from PostgreSQL (written by PersistenceActor during trading)
and candle data from the ParquetDataCatalog.

**Prerequisites:**
- PostgreSQL running (`docker compose up -d postgres`)
- At least one completed or active trading run with fills in the database
- Candle data in `data/catalog/` covering the run's time period

In [ ]:
# ── Cell 1: Imports + config ───────────────────────────────────────

import asyncio
from decimal import Decimal
from pathlib import Path

import asyncpg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.config.settings import get_settings
from src.core import bar_type_str
from charts import generate_backtest_html, plot_pnl_heatmap

settings = get_settings()
CATALOG_PATH = "../data/catalog"
BAR_INTERVAL = "5m"  # Match what the strategy was trading on
# Infer MA periods from strategy config if available
# Chart uses these for overlay lines only, not for signal generation
FAST_PERIOD = 5
SLOW_PERIOD = 45
MA_TYPE = "EMA"  # or "SMA" depending on your strategy

print(f"DB: {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print("Imports OK")

In [ ]:
# ── Cell 2: List recent runs ───────────────────────────────────────

async def _list_runs():
    conn = await asyncpg.connect(settings.postgres_dsn)
    try:
        rows = await conn.fetch("""
            SELECT
                r.id,
                r.strategy_id,
                r.instrument_id,
                r.run_mode,
                r.started_at,
                r.stopped_at,
                (SELECT COUNT(*) FROM order_fills f WHERE f.run_id = r.id) AS fills,
                (SELECT COUNT(*) FROM positions p WHERE p.run_id = r.id) AS positions
            FROM strategy_runs r
            ORDER BY r.started_at DESC
            LIMIT 20
        """)
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()

runs_df = await _list_runs()

if runs_df.empty:
    print("No runs found in database.")
else:
    print(f"Found {len(runs_df)} recent runs:\n")
    for i, row in runs_df.iterrows():
        status = "running" if row["stopped_at"] is None else "stopped"
        print(
            f"  [{i}] {row['strategy_id']}  {row['instrument_id']}  "
            f"{row['run_mode']}  {status}  "
            f"fills={row['fills']}  positions={row['positions']}  "
            f"started={row['started_at']:%Y-%m-%d %H:%M}"
        )

In [ ]:
# ── Cell 3: Select run + load data ─────────────────────────────────
#
# Change PICK to select a different run from the list above.

PICK = 0

run = runs_df.iloc[PICK]
RUN_ID = run["id"]
STRATEGY_ID = run["strategy_id"]
INSTRUMENT_ID = run["instrument_id"]
RUN_MODE = run["run_mode"]

print(f"Selected run: {STRATEGY_ID} on {INSTRUMENT_ID} ({RUN_MODE})")
print(f"Run ID: {RUN_ID}")
print(f"Started: {run['started_at']}")
print(f"Stopped: {run['stopped_at'] or 'still running'}")
print(f"Fills: {run['fills']}, Positions: {run['positions']}")


async def _load_run_data(run_id):
    conn = await asyncpg.connect(settings.postgres_dsn)
    try:
        fills_rows = await conn.fetch("""
            SELECT ts, strategy_id, instrument_id, client_order_id,
                   order_side, last_qty, last_px, commission,
                   commission_currency, liquidity_side
            FROM order_fills
            WHERE run_id = $1
            ORDER BY ts
        """, run_id)

        positions_rows = await conn.fetch("""
            SELECT ts_opened, ts_closed, strategy_id, instrument_id,
                   position_id, entry_side, peak_qty,
                   avg_px_open, avg_px_close,
                   realized_pnl, realized_return, currency, duration_ns
            FROM positions
            WHERE run_id = $1
            ORDER BY ts_closed
        """, run_id)

        snapshots_rows = await conn.fetch("""
            SELECT ts, venue, currency, balance_total, balance_free, balance_locked
            FROM account_snapshots
            WHERE run_id = $1
            ORDER BY ts
        """, run_id)

        return (
            pd.DataFrame([dict(r) for r in fills_rows]),
            pd.DataFrame([dict(r) for r in positions_rows]),
            pd.DataFrame([dict(r) for r in snapshots_rows]),
        )
    finally:
        await conn.close()

raw_fills, raw_positions, raw_snapshots = await _load_run_data(RUN_ID)
print(f"\nLoaded: {len(raw_fills)} fills, {len(raw_positions)} positions, {len(raw_snapshots)} snapshots")

In [ ]:
# ── Cell 4: Reshape data for charts.py ─────────────────────────────
#
# charts.py functions expect specific column names from NT's in-memory
# reports. PG schema uses slightly different names. Map them here.

# ── Fills ──
# _parse_fills looks for: last_px, order_side, ts_last (int64 ns), last_qty
# _fills_to_markers looks for: ts_last, side, avg_px, filled_qty
fills_df = raw_fills.copy()
if not fills_df.empty:
    # Convert TIMESTAMPTZ to nanosecond int64 (what NT uses internally)
    fills_df["ts_last"] = pd.to_datetime(fills_df["ts"], utc=True).astype("int64")
    # Add column aliases for charts.py compatibility
    fills_df["side"] = fills_df["order_side"]
    fills_df["avg_px"] = fills_df["last_px"]
    fills_df["filled_qty"] = fills_df["last_qty"]
    print(f"Fills: {len(fills_df)} rows, {fills_df['ts'].min():%Y-%m-%d} → {fills_df['ts'].max():%Y-%m-%d}")
else:
    print("No fills found.")

# ── Positions ──
# _positions_to_rows looks for: ts_opened, ts_closed, entry, peak_qty,
#   avg_px_open, avg_px_close, realized_pnl, realized_return
positions_df = raw_positions.copy()
if not positions_df.empty:
    # Rename entry_side → entry (charts.py expects "entry")
    positions_df["entry"] = positions_df["entry_side"]
    # realized_pnl is NUMERIC in PG — _parse_money_str handles plain numbers
    print(f"Positions: {len(positions_df)} closed trades")
else:
    print("No closed positions found.")

# ── Account snapshots ──
snapshots_df = raw_snapshots.copy()
if not snapshots_df.empty:
    snapshots_df["ts"] = pd.to_datetime(snapshots_df["ts"], utc=True)
    print(f"Snapshots: {len(snapshots_df)} balance records")
else:
    print("No account snapshots found.")

In [ ]:
# ── Cell 5: Load bars from catalog ─────────────────────────────────
#
# The TVLC chart needs candle data. Load from catalog for the run's
# time period. If bars aren't available, run your fetch script first:
#   python scripts/fetch_hl_candles.py --coins BTC --intervals 1h

BAR_TYPE_STR = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)

catalog = ParquetDataCatalog(CATALOG_PATH)

try:
    bars = catalog.bars(bar_types=[BAR_TYPE_STR])
    print(f"Loaded {len(bars):,} bars for {BAR_TYPE_STR}")

    # Filter to run's time range (with some padding for chart context)
    if not fills_df.empty:
        run_start = fills_df["ts"].min()
        run_end = fills_df["ts"].max()

        # Add padding: 5% of run duration on each side for chart context
        run_duration = run_end - run_start
        pad = run_duration * 0.05
        filter_start = int((run_start - pad).timestamp() * 1e9)
        filter_end = int((run_end + pad).timestamp() * 1e9)

        bars = [b for b in bars if filter_start <= b.ts_event <= filter_end]
        print(f"Filtered to {len(bars):,} bars covering run period")

except Exception as e:
    bars = []
    print(f"Could not load bars: {e}")
    print(f"Run your fetch script to get candle data for {INSTRUMENT_ID}")

In [ ]:
# ── Cell 6: Equity curve from account snapshots ────────────────────

if not snapshots_df.empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor("#131722")
    ax.set_facecolor("#131722")

    ax.plot(
        snapshots_df["ts"],
        snapshots_df["balance_total"].astype(float),
        color="#2196f3",
        linewidth=1.5,
    )

    # Starting balance reference line
    start_balance = float(snapshots_df["balance_total"].iloc[0])
    ax.axhline(start_balance, color="#787b86", linewidth=0.5, linestyle="--", alpha=0.5)

    ax.set_title(
        f"Equity Curve — {STRATEGY_ID} on {INSTRUMENT_ID} ({RUN_MODE})",
        color="#d1d4dc", fontsize=13,
    )
    ax.set_xlabel("Time", color="#d1d4dc")
    ax.set_ylabel(f"Balance ({snapshots_df['currency'].iloc[0]})", color="#d1d4dc")
    ax.tick_params(colors="#d1d4dc")
    ax.grid(True, alpha=0.1)

    # Annotate final balance
    final_balance = float(snapshots_df["balance_total"].iloc[-1])
    pnl = final_balance - start_balance
    pnl_pct = pnl / start_balance * 100
    color = "#26a69a" if pnl >= 0 else "#ef5350"
    ax.annotate(
        f"  {final_balance:,.2f} ({pnl:+,.2f} / {pnl_pct:+.1f}%)",
        xy=(snapshots_df["ts"].iloc[-1], final_balance),
        color=color, fontsize=11, fontweight="bold",
        va="center",
    )

    plt.tight_layout()
    plt.show()
else:
    print("No account snapshots — equity curve not available.")

In [ ]:
# ── Cell 7: Trade stats summary ───────────────────────────────────

if not positions_df.empty:
    pnls = positions_df["realized_pnl"].astype(float)
    returns = positions_df["realized_return"].astype(float)

    winners = pnls[pnls > 0]
    losers = pnls[pnls < 0]
    gross_win = winners.sum()
    gross_loss = abs(losers.sum())

    currency = positions_df["currency"].iloc[0] or "USD"

    print(f"{'═' * 50}")
    print(f"  TRADE SUMMARY — {STRATEGY_ID} ({RUN_MODE})")
    print(f"  {INSTRUMENT_ID}")
    print(f"{'═' * 50}")
    print(f"  Total trades:    {len(pnls)}")
    print(f"  Winners:         {len(winners)} ({len(winners)/len(pnls)*100:.0f}%)")
    print(f"  Losers:          {len(losers)} ({len(losers)/len(pnls)*100:.0f}%)")
    print(f"  Total PnL:       {pnls.sum():,.2f} {currency}")
    print(f"  Avg winner:      {winners.mean():,.2f} {currency}" if len(winners) > 0 else "")
    print(f"  Avg loser:       {losers.mean():,.2f} {currency}" if len(losers) > 0 else "")
    print(f"  Profit factor:   {gross_win / gross_loss:.2f}" if gross_loss > 0 else "  Profit factor:   ∞")
    print(f"  Avg return:      {returns.mean()*100:.2f}%")
    print(f"  Best trade:      {pnls.max():,.2f} {currency}")
    print(f"  Worst trade:     {pnls.min():,.2f} {currency}")

    if not snapshots_df.empty:
        start_bal = float(snapshots_df["balance_total"].iloc[0])
        final_bal = float(snapshots_df["balance_total"].iloc[-1])
        print(f"  Starting bal:    {start_bal:,.2f} {currency}")
        print(f"  Final bal:       {final_bal:,.2f} {currency}")
        print(f"  Return on acct:  {(final_bal - start_bal) / start_bal * 100:+.2f}%")

    print(f"{'═' * 50}")
else:
    print("No closed positions — no stats available.")

In [ ]:
# ── Cell 8: PnL per trade bar chart ───────────────────────────────

if not positions_df.empty:
    pnls = positions_df["realized_pnl"].astype(float).values

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#131722")

    # Left: PnL per trade (bar chart)
    ax = axes[0]
    ax.set_facecolor("#131722")
    colors = ["#26a69a" if p > 0 else "#ef5350" for p in pnls]
    ax.bar(range(len(pnls)), pnls, color=colors, width=1.0, edgecolor="none")
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("Trade #", color="#d1d4dc")
    ax.set_ylabel("PnL", color="#d1d4dc")
    ax.set_title("PnL Per Trade", color="#d1d4dc")
    ax.tick_params(colors="#d1d4dc")

    # Right: Cumulative PnL
    ax = axes[1]
    ax.set_facecolor("#131722")
    cum_pnl = np.cumsum(pnls)
    ax.plot(cum_pnl, color="#2196f3", linewidth=1.5)
    ax.fill_between(
        range(len(cum_pnl)), cum_pnl, 0,
        where=cum_pnl >= 0, color="#26a69a", alpha=0.15,
    )
    ax.fill_between(
        range(len(cum_pnl)), cum_pnl, 0,
        where=cum_pnl < 0, color="#ef5350", alpha=0.15,
    )
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("Trade #", color="#d1d4dc")
    ax.set_ylabel("Cumulative PnL", color="#d1d4dc")
    ax.set_title("Cumulative PnL", color="#d1d4dc")
    ax.tick_params(colors="#d1d4dc")

    plt.tight_layout()
    plt.show()
else:
    print("No positions to plot.")

In [ ]:
# ── Cell 9: TradingView Lightweight Charts HTML report ─────────────
#
# Same interactive chart as the backtest notebook — candlesticks with
# buy/sell markers, stats bar, trade table with click-to-zoom.

if bars and not fills_df.empty:

    run_start_str = f"{run['started_at']:%Y%m%d}"
    output_path = (
        f"../reports/live_{STRATEGY_ID}_{INSTRUMENT_ID}_{run_start_str}.html"
    )

    # Determine starting capital from first account snapshot
    starting_capital = (
        float(snapshots_df["balance_total"].iloc[0])
        if not snapshots_df.empty
        else 10_000
    )

    path = generate_backtest_html(
        bars,
        fills_df,
        positions_df,
        fast_period=FAST_PERIOD,
        slow_period=SLOW_PERIOD,
        ma_type=MA_TYPE,
        instrument_label=INSTRUMENT_ID,
        bar_label=BAR_INTERVAL,
        starting_capital=starting_capital,
        output_path=output_path,
    )
    print(f"Open in browser: {path}")
elif not bars:
    print("No bar data available. Run your candle fetch script first:")
    print(f"  python scripts/fetch_hl_candles.py --coins {INSTRUMENT_ID.split('-')[0]}")
else:
    print("No fills to overlay on chart.")